In [0]:
student_data = [
    (1, "Daniel"),
    (2, "Jade"),
    (3, "Stella"),
    (4, "Jonathan"),
    (5, "Will")
]
student_df = spark.createDataFrame(student_data, ["student_id", "student_name"])

exam_data = [
    (10, 1, 70),
    (10, 2, 80),
    (10, 3, 90),
    (20, 1, 80),
    (30, 1, 70),
    (30, 3, 80),
    (30, 4, 90),
    (40, 1, 60),
    (40, 2, 70),
    (40, 4, 80)
]
exam_df = spark.createDataFrame(exam_data, ["exam_id", "student_id", "score"])

display(student_df)

display(exam_df)

In [0]:
student_df.createOrReplaceTempView('student')
exam_df.createOrReplaceTempView('exam')

In [0]:
spark.sql("""
    with cte as (
    SELECT student_id, exam_id, score,
           RANK() OVER (
                PARTITION BY exam_id
                ORDER BY score
            ) AS rk1,
            RANK() OVER (
                PARTITION BY exam_id
                ORDER BY score DESC
            ) AS rk2
        FROM Exam
          )
    select student_id from cte
    group by student_id
    having (SUM(CASE WHEN rk1=1 THEN 1 ELSE 0 END) = 0) AND (SUM(CASE WHEN rk2=1 THEN 1 ELSE 0 END) = 0)
""").display()